In [21]:
import pandas as pd
from IPython.core.interactiveshell import InteractiveShell
from rdkit import Chem

import stereomolgraph.experimental._sym_num as sym_num
from stereomolgraph import StereoMolGraph
from stereomolgraph.experimental._pandas_patcher import patchPandas
from stereomolgraph.ipython import View2D

patchPandas()

InteractiveShell.ast_node_interactivity = "all"
View2D.show_h = False

In [22]:
inchis_kinbot_fail = [
    "InChI=1S/C16H34/c1-3-5-7-9-11-13-15-16-14-12-10-8-6-4-2/h3-16H2,1-2H3",
    "InChI=1S/C18H12/c1-2-8-14-13(7-1)15-9-3-4-11-17(15)18-12-6-5-10-16(14)18/h1-12H",
    "InChI=1S/C4H5/c1-3-4-2/h1H2,2H3",
    "InChI=1S/C7H7/c1-7-5-3-2-4-6-7/h2-6H,1H2",
    "InChI=1S/C7H8/c1-2-4-5(2)7-3(1)6(4)7/h2-7H,1H2",
    "InChI=1S/C8H14/c1-2-8-5-3-7(1)4-6-8/h7-8H,1-6H2",
    "InChI=1S/C8H8/c1-2-5-3(1)7-4(1)6(2)8(5)7/h1-8H",
]

In [23]:
inchis_andreas_mail = [
    "InChI=1S/C7H16O4/c1-4(8)7(11,5(2)9)6(3)10/h4-6,8-11H,1-3H3/t4-,5-,6+/m0/s1",
    "InChI=1S/C7H16O4/c1-4(8)7(11,5(2)9)6(3)10/h4-6,8-11H,1-3H3/t4-,5-,6-/m0/s1",
    "InChI=1S/C10H18O/c1-2-9-3-6-10(11,7-4-9)8-5-9/h11H,2-8H2,1H3",
    "InChI=1S/CH5N/c1-2/h2H2,1H3",
    "InChI=1S/C2H5/c1-2/h1H2,2H3"
]

In [24]:
inchis_input = inchis_andreas_mail

rows = []
for inchi in inchis_input:
    rdmol = Chem.MolFromInchi(inchi)
    if rdmol is None:
        rows.append(
            {
                "inchi": inchi,
                "stereomolgraph": None,
                "top_sym_num": None,
                "rotational_symmetry_number": None,
            }
        )
        continue

    rdmol = Chem.AddHs(rdmol)
    graph = StereoMolGraph.from_rdmol(rdmol, stereo_complete=True)

    working_by_bond = sym_num.working_bond_symmetry_numbers(graph)
    classes_by_symmetry = sym_num.bond_automorphism_classes_by_symmetry(
        graph,
        working_by_bond=working_by_bond,
    )

    row = {
        "inchi": inchi,
        "stereomolgraph": graph,
        "top_sym_num": sym_num.topological_symmetry_number(graph),
        #"rotational_symmetry_number": None,  # external_symmetry_number(graph),
    }

    for sym_value, bond_classes in sorted(classes_by_symmetry.items()):
        row[f"bond_symmetry_{sym_value}"] = bond_classes

    rows.append(row)

df_symmetry = pd.DataFrame(rows)

symmetry_columns = sorted(
    (column for column in df_symmetry.columns if column.startswith("bond_symmetry_")),
    key=lambda column: int(column.removeprefix("bond_symmetry_")),
)

base_columns = [
    "inchi",
    "stereomolgraph",
    "top_sym_num",
    #"rotational_symmetry_number",
]

df_symmetry = df_symmetry.reindex(columns=base_columns + symmetry_columns)
df_symmetry

inchi  \
0  InChI=1S/C7H16O4/c1-4(8)7(11,5(2)9)6(3)10/h4-6...   
1  InChI=1S/C7H16O4/c1-4(8)7(11,5(2)9)6(3)10/h4-6...   
2  InChI=1S/C10H18O/c1-2-9-3-6-10(11,7-4-9)8-5-9/...   
3                        InChI=1S/CH5N/c1-2/h2H2,1H3   
4                        InChI=1S/C2H5/c1-2/h1H2,2H3   

                                      stereomolgraph  top_sym_num  \
0  StereoMolGraph\nAtoms\n(0 C) (1 C) (2 C) (3 C)...           27   
1  StereoMolGraph\nAtoms\n(0 C) (1 C) (2 C) (3 C)...           81   
2  StereoMolGraph\nAtoms\n(0 C) (1 C) (2 C) (3 C)...            9   
3  StereoMolGraph\nAtoms (0 C) (1 N) (2 H) (3 H) ...            6   
4  StereoMolGraph\nAtoms (0 C) (1 C) (2 H) (3 H) ...            6   

                                     bond_symmetry_1  \
0  (((0, 11), (0, 12), (0, 13)), ((1, 14), (1, 15...   
1  (((0, 11), (0, 12), (0, 13), (1, 14), (1, 15),...   
2  (((0, 11), (0, 12), (0, 13)), ((1, 14),), ((1,...   
3       (((0, 2), (0, 3), (0, 4)), ((1, 5), (1, 6)))   
4       (((0, 2), (0, 3)), ((1, 4), (1, 5), (1, 6)))   

                          bond_symmetry_3 bond_symmetry_6  
0       (((0, 3),), ((1, 4),), ((2, 5),))             NaN  
1  (((0, 3), (1, 4), (2, 5)), ((6, 10),))             NaN  
2      (((0, 1),), ((1, 8),), ((9, 10),))             NaN  
3                                     NaN    (((0, 1),),)  
4                                     NaN    (((0, 1),),)